<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week5_first_classifier_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 5 — Teaching the Model to Read (STARTER)
### The NLP Internship | LinguaAI Labs

This week: your first text classifier on **real customer intent data** from CLINC150.

**Before any code — write your primary metric and deployment threshold.**

## My Metric Decision

**Primary metric:** [WRITE HERE]
**Reasoning:** [WRITE HERE]
**Deployment threshold:** [WRITE HERE]

In [ ]:
import sys, subprocess, os
for pkg in ["datasets","scikit-learn","matplotlib","seaborn"]:
    try: __import__(pkg.replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
from datasets import load_dataset
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42); os.makedirs("outputs", exist_ok=True)

DOMAIN_MAP = {
    "bill_balance":"billing","bill_pay":"billing","transfer":"billing",
    "pay_bill":"billing","min_payment":"billing","account_blocked":"billing",
    "interest_rate":"billing","routing":"billing","spending_history":"billing",
    "credit_score":"account","credit_limit":"account","damaged_card":"account",
    "freeze_account":"account","replacement_card":"account","pin_change":"account",
    "lost_stolen_card":"account","cancel_card":"account","card_declined":"account",
    "report_fraud":"account","expiration_date":"account",
    "travel_alert":"travel","flight_status":"travel","travel_suggestion":"travel",
    "book_hotel":"travel","book_flight":"travel","exchange_rate":"travel",
    "reminder":"utility","reminder_update":"utility","calendar":"utility",
    "schedule_meeting":"utility","make_call":"utility","text":"utility",
    "timer":"utility","alarm":"utility","tell_joke":"utility",
}
clinc = load_dataset("clinc_oos", "plus")
label_names = clinc["train"].features["intent"].names
def collapse(i): return DOMAIN_MAP.get(label_names[i], "other")
df_train = clinc["train"].to_pandas().rename(columns={"text":"text_clean"})
df_test  = clinc["test"].to_pandas().rename(columns={"text":"text_clean"})
df_train["category"] = df_train["intent"].apply(collapse)
df_test["category"]  = df_test["intent"].apply(collapse)
X_train=df_train["text_clean"].values; y_train=df_train["category"].values
X_test =df_test["text_clean"].values;  y_test =df_test["category"].values
print(f"CLINC150: {len(X_train):,} train | {len(X_test):,} test | {df_train['category'].nunique()} domains ✅")

## Part 1 — Inspect the Real Data

> ⏸️ **Pause and Predict**
> Which domain will be largest? Is CLINC150 more or less balanced than a typical support ticket corpus?

In [ ]:
print("Domain distribution (train):\n")
for cat, count in df_train["category"].value_counts().items():
    bar = "█" * (count // 50)
    print(f"  {cat:<22} {count:>5,} ({count/len(df_train):.1%})  {bar}")
print(f"\nSample queries per domain:")
for domain in df_train["category"].unique():
    s = df_train[df_train["category"]==domain]["text_clean"].iloc[0]
    print(f"  [{domain}] {s}")

## Part 2 — DummyClassifier Baseline

> 🧠 **Predict the weighted F1 before running.**

In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
dummy_f1 = f1_score(y_test, dummy.predict(X_test), average="weighted", zero_division=0)
print(f"Dummy F1: {dummy_f1:.3f}  ← the floor. Beat it.")
print(classification_report(y_test, dummy.predict(X_test), zero_division=0))

## Part 3 — TF-IDF + Logistic Regression

> ⏸️ **Predict:** will short CLINC queries hurt TF-IDF more or less than long support tickets?

In [ ]:
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(multi_class="multinomial", solver="lbfgs", max_iter=1000,
                                C=1.0, class_weight=None,  # YOUR CODE HERE — try "balanced"
                                random_state=42)),
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
lr_f1 = f1_score(y_test, y_pred_lr, average="weighted", zero_division=0)
print(f"TF-IDF + LR: {lr_f1:.3f}  (dummy: {dummy_f1:.3f})")
print(classification_report(y_test, y_pred_lr, zero_division=0))
# ⚠️ Label Leakage: Pipeline fits vectoriser on training data only — correct.

## Part 4 — Naive Bayes Comparison

In [ ]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=False)),
    ("clf", MultinomialNB(alpha=0.1)),
])
nb_pipeline.fit(X_train, y_train)
y_pred_nb = nb_pipeline.predict(X_test)
nb_f1 = f1_score(y_test, y_pred_nb, average="weighted", zero_division=0)
print(f"NB: {nb_f1:.3f}  LR: {lr_f1:.3f}  Dummy: {dummy_f1:.3f}")
# YOUR CODE HERE — which is better and why? Write in a markdown cell.

## Part 5 — Error Analysis

In [ ]:
classes = lr_pipeline.named_steps["clf"].classes_
cm = confusion_matrix(y_test, y_pred_lr, labels=classes)
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(pd.DataFrame(cm, index=classes, columns=classes),
            annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix — TF-IDF + LR on CLINC150", fontweight="bold", loc="left")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig("outputs/confusion_matrix_clinc.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
errors = pd.DataFrame({"text":X_test,"true":y_test,"pred":y_pred_lr})
errors = errors[errors["true"] != errors["pred"]]
print(f"Errors: {len(errors)}/{len(y_test)} ({len(errors)/len(y_test):.1%})")
print(errors.groupby(["true","pred"]).size().sort_values(ascending=False).head(6))

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Using accuracy on imbalanced text data | If 80% of tickets are 'billing', a classifier that always predicts 'billing' scores 80%. That classifier is useless. | Always report precision, recall, and F1 per class alongside overall accuracy |
| Vectorising before splitting | If you fit TF-IDF on the full dataset, test set vocabulary leaks into training. Split first, vectorise after. | train_test_split → fit vectoriser on X_train → transform X_train and X_test |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Train a Logistic Regression classifier on TF-IDF features. Report precision, recall, and F1 per class. Which class is hardest to classify?

> 🔬 **Advanced Path**
> Train both Logistic Regression and Naive Bayes. Compare per-class F1 scores. Which performs better on minority classes, and why?

## ✅ What You Can Do After This Week

- Load and inspect CLINC150, a real published NLP dataset
- Build a TF-IDF + LR pipeline with correct train/test separation
- Compare LR vs Naive Bayes and explain the tradeoff
- Conduct structured error analysis on a real multi-class classifier

---
## ✅ Week 5 Complete
**Next:** `week6_word_vectors_STARTER.ipynb`

---
*Add `week5_first_classifier_STARTER.ipynb` to your internship portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 5, you can now:

- Frame a text classification problem and justify the choice of model family
- Vectorise text correctly — fitting on training data only
- Evaluate a classifier using precision, recall, and F1 per class
- Explain why accuracy alone is the wrong metric for imbalanced text data

📤 **GitHub:** Push week5_first_classifier_STARTER.ipynb and model_card_v1.md. Commit: "Week 5: First classifier — TF-IDF + LR with evaluation"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
